# 🏦 Loan Approval Prediction — End-to-End ML Pipeline

**Dataset:** Dream Housing Finance Loan Prediction  
**Task:** Binary Classification — predict whether a loan application will be approved (`Y`) or rejected (`N`)  
**Target Column:** `Loan_Status`

---
### Pipeline Overview
`Load Data` → `Explore` → `Clean & Preprocess` → `Train Models` → `Evaluate` → `Tune Best Model` → `Save`


## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid", palette="muted")

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

print("✅ All libraries imported successfully.")

## 2. 📂 Load the Dataset

In [ ]:
# ── File paths ── (update if running locally)
TRAIN_PATH = "train_u6lujuX_CVtuZ9i.csv"
TEST_PATH  = "test_Y3wMUE5_7gLdaTN.csv"

# Google Colab: upload files or mount Drive, then set correct paths above.
# If files are in the same directory as this notebook, no change needed.

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train shape : {df_train.shape}")
print(f"Test  shape : {df_test.shape}")
df_train.head()

## 3. 🔍 Exploratory Data Analysis

### 3.1 Shape, Columns & Data Types

In [ ]:
print("─── Columns & dtypes ───")
print(df_train.dtypes)
print(f"\nRows: {df_train.shape[0]}  |  Columns: {df_train.shape[1]}")

### 3.2 Summary Statistics

In [ ]:
df_train.describe(include="all").T

### 3.3 Missing Values

In [ ]:
missing = df_train.isnull().sum()
missing_pct = (missing / len(df_train) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)
print(missing_df)

# Visual
fig, ax = plt.subplots(figsize=(8, 4))
missing_df["Missing %"].plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Missing Values (%)", fontsize=13, fontweight="bold")
ax.set_ylabel("Percentage")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.show()

### 3.4 Duplicate Rows

In [ ]:
n_dupes = df_train.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

### 3.5 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df_train["Loan_Status"].value_counts()
axes[0].pie(counts, labels=["Approved (Y)", "Rejected (N)"],
            autopct="%1.1f%%", colors=["#4CAF50", "#F44336"], startangle=90)
axes[0].set_title("Loan Status Distribution")

sns.countplot(data=df_train, x="Loan_Status", palette={"Y": "#4CAF50", "N": "#F44336"}, ax=axes[1])
axes[1].set_title("Loan Status Count")
axes[1].set_xlabel("Loan Status")
plt.tight_layout()
plt.show()
print(counts)

### 3.6 Feature Distributions & Relationships

In [ ]:
# Numeric feature distributions
num_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), num_cols):
    sns.histplot(data=df_train, x=col, hue="Loan_Status", kde=True, ax=ax,
                 palette={"Y": "#4CAF50", "N": "#F44336"}, alpha=0.6)
    ax.set_title(f"{col} vs Loan Status")
plt.suptitle("Numeric Features vs Loan Status", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features
cat_cols = ["Gender", "Married", "Education", "Self_Employed", "Property_Area", "Credit_History"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    ct = df_train.groupby([col, "Loan_Status"]).size().unstack(fill_value=0)
    ct.plot(kind="bar", ax=ax, color=["#F44336", "#4CAF50"], edgecolor="white")
    ax.set_title(f"{col} vs Loan Status")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Categorical Features vs Loan Status", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric only)
corr_df = df_train.select_dtypes(include="number")
plt.figure(figsize=(8, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. 🧹 Data Cleaning & Preprocessing

### 4.1 Drop ID Column & Remove Duplicates

In [ ]:
df = df_train.copy()

# Drop identifier — no predictive value
df.drop(columns=["Loan_ID"], inplace=True)

# Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {before - len(df)} duplicate row(s). Remaining: {len(df)}")

### 4.2 Encode Target Column

In [ ]:
# Map target: Y → 1, N → 0
TARGET = "Loan_Status"
df[TARGET] = df[TARGET].map({"Y": 1, "N": 0})
print("Target encoding:  Y=1 (Approved), N=0 (Rejected)")
print(df[TARGET].value_counts())

### 4.3 Define Feature Groups

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Identify column types
categorical_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_cols   = X.select_dtypes(include=["number"]).columns.tolist()

print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical features   ({len(numerical_cols)}): {numerical_cols}")

### 4.4 Build Preprocessing Pipelines

We use `sklearn.pipeline.Pipeline` + `ColumnTransformer` for a clean, leak-free preprocessing:

| Step | Numerical | Categorical |
|------|-----------|-------------|
| Impute | Median | Most Frequent |
| Scale/Encode | `StandardScaler` | `OneHotEncoder` |


In [ ]:
# Numerical pipeline: impute with median → standard-scale
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

# Categorical pipeline: impute with mode → one-hot encode
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Combined transformer
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])

print("✅ Preprocessing pipelines defined.")

### 4.5 Save the Cleaned Dataset

In [ ]:
CLEANED_FILE = "loan_cleaned_dataset.csv"
df.to_csv(CLEANED_FILE, index=False)
print(f"✅ Cleaned dataset saved → '{CLEANED_FILE}'  ({df.shape[0]} rows × {df.shape[1]} cols)")

## 5. ✂️ Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train)}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing  samples : {len(X_test)}  ({len(X_test)/len(X)*100:.1f}%)")
print(f"Class balance (train) — 1: {y_train.sum()}  |  0: {(y_train==0).sum()}")

## 6. 🤖 Train Multiple Models

We compare four classifiers wrapped inside full `Pipeline` objects (preprocessing + model):

In [ ]:
# Define models
models = {
    "Logistic Regression":    LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":          DecisionTreeClassifier(random_state=42),
    "Random Forest":          RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting":      GradientBoostingClassifier(n_estimators=200, random_state=42),
}

# Build full pipelines
pipelines = {
    name: Pipeline([("preprocessor", preprocessor), ("model", clf)])
    for name, clf in models.items()
}

print("✅ Pipelines ready for:", list(pipelines.keys()))

## 7. 📊 Model Evaluation

### 7.1 Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<25}  CV F1 = {scores.mean():.4f} ± {scores.std():.4f}")

### 7.2 Train on Full Training Set & Evaluate on Hold-Out Test Set

In [ ]:
results = []

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    cv_f1_mean = cv_results[name].mean()

    results.append({
        "Model": name,
        "Accuracy":    round(acc, 4),
        "F1 Score":    round(f1, 4),
        "Precision":   round(prec, 4),
        "Recall":      round(rec, 4),
        "CV F1 (mean)": round(cv_f1_mean, 4),
    })

results_df = pd.DataFrame(results).set_index("Model").sort_values("F1 Score", ascending=False)
print(results_df.to_string())

### 7.3 Comparison Chart

In [ ]:
metrics = ["Accuracy", "F1 Score", "Precision", "Recall"]
x = np.arange(len(results_df))
width = 0.2
colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]

fig, ax = plt.subplots(figsize=(13, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i * width, results_df[metric], width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df.index, rotation=15, ha="right", fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Classification Metrics", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.axhline(0.80, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

### 7.4 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (name, pipe) in zip(axes.flatten(), pipelines.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Rejected (0)", "Approved (1)"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}  (F1={f1_score(y_test,y_pred):.3f})", fontsize=10, fontweight="bold")
plt.suptitle("Confusion Matrices — All Models", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 8. 🌲 Feature Importance (Random Forest & Gradient Boosting)

In [ ]:
def get_feature_names(preprocessor, num_cols, cat_cols):
    ohe = preprocessor.named_transformers_["cat"]["encoder"]
    cat_names = ohe.get_feature_names_out(cat_cols).tolist()
    return num_cols + cat_names

feature_names = get_feature_names(preprocessor, numerical_cols, categorical_cols)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, name in zip(axes, ["Random Forest", "Gradient Boosting"]):
    model = pipelines[name].named_steps["model"]
    importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=True)
    importances.tail(15).plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"{name} — Top 15 Features", fontsize=11, fontweight="bold")
    ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## 8b. 📈 Logistic Regression Coefficients

In [ ]:
lr_model = pipelines["Logistic Regression"].named_steps["model"]
lr_names = get_feature_names(
    pipelines["Logistic Regression"].named_steps["preprocessor"],
    numerical_cols, categorical_cols
)
coef_series = pd.Series(lr_model.coef_[0], index=lr_names).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors_lr = ["#F44336" if c < 0 else "#4CAF50" for c in coef_series]
coef_series.plot(kind="bar", ax=ax, color=colors_lr, edgecolor="white")
ax.set_title("Logistic Regression — Coefficients", fontsize=12, fontweight="bold")
ax.set_ylabel("Coefficient Value")
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## 9. 🔧 Hyperparameter Tuning with GridSearchCV

We tune the **Random Forest** (best CV F1 score) using `GridSearchCV` with stratified 5-fold CV.


In [ ]:
param_grid = {
    "model__n_estimators":      [100, 200, 300],
    "model__max_depth":         [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__max_features":      ["sqrt", "log2"],
}

base_rf_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(
    estimator  = base_rf_pipe,
    param_grid = param_grid,
    cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring    = "f1",
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Best CV F1 Score : {grid_search.best_score_:.4f}")
print(f"   Best Parameters  : {grid_search.best_params_}")

In [ ]:
# Evaluate tuned model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

best_acc  = accuracy_score(y_test, y_pred_best)
best_f1   = f1_score(y_test, y_pred_best)
best_prec = precision_score(y_test, y_pred_best)
best_rec  = recall_score(y_test, y_pred_best)

print("\n── Tuned Random Forest — Test Set Performance ──")
print(f"  Accuracy  : {best_acc:.4f}")
print(f"  F1 Score  : {best_f1:.4f}")
print(f"  Precision : {best_prec:.4f}")
print(f"  Recall    : {best_rec:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=["Rejected", "Approved"]))

In [ ]:
# Confusion matrix for best model
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(cm, display_labels=["Rejected", "Approved"]).plot(ax=ax, cmap="Greens", colorbar=False)
ax.set_title("Tuned Random Forest — Confusion Matrix", fontweight="bold")
plt.tight_layout()
plt.show()

## 10. 💾 Save the Best Model

In [ ]:
MODEL_FILE = "best_loan_approval_model.pkl"
joblib.dump(best_model, MODEL_FILE)
print(f"✅ Best model saved → '{MODEL_FILE}'")

# Quick reload test
reloaded = joblib.load(MODEL_FILE)
reload_preds = reloaded.predict(X_test[:5])
print(f"   Reload test (first 5 predictions): {reload_preds}")

## 11. 🚀 Predict on the Original Held-Out Test File

In [ ]:
df_holdout = pd.read_csv(TEST_PATH)
loan_ids   = df_holdout["Loan_ID"]
X_holdout  = df_holdout.drop(columns=["Loan_ID"])

preds_numeric = best_model.predict(X_holdout)
preds_label   = ["Y" if p == 1 else "N" for p in preds_numeric]

submission = pd.DataFrame({"Loan_ID": loan_ids, "Loan_Status": preds_label})
SUBMISSION_FILE = "loan_submission_predictions.csv"
submission.to_csv(SUBMISSION_FILE, index=False)

print(f"✅ Predictions saved → '{SUBMISSION_FILE}'")
print(submission["Loan_Status"].value_counts())
submission.head(10)

---
## 12. 🏆 Best Model Summary

| Item | Value |
|------|-------|
| **Best Model** | Tuned Random Forest Classifier |
| **Best CV F1 Score (GridSearchCV)** | See `grid_search.best_score_` output above |
| **Test Accuracy** | See evaluation output above |
| **Test F1 Score** | See evaluation output above |
| **Saved Model File** | `best_loan_approval_model.pkl` |
| **Cleaned Dataset File** | `loan_cleaned_dataset.csv` |
| **Submission Predictions** | `loan_submission_predictions.csv` |

### Why Random Forest Performed Best

1. **Ensemble power** — averages hundreds of decorrelated decision trees, reducing variance and overfitting.
2. **Handles mixed data well** — copes naturally with the numerical + categorical mix in this dataset.
3. **Robust to outliers** — `ApplicantIncome` and `CoapplicantIncome` have heavy right skew; tree splits are insensitive to outlier magnitude.
4. **Built-in feature selection** — `max_features` sampling per split prevents dominant features from drowning weaker predictors like `Dependents`.
5. **Credit History signal** — the most decisive feature (confirmed in importance plot); Random Forest picks this up reliably without manual feature engineering.

### Key Insight

`Credit_History` is overwhelmingly the strongest predictor, followed by income and loan amount ratios. Applicants with `Credit_History = 1` have ~80% approval rates, which aligns with real-world lending behaviour.


In [ ]:
print("=" * 55)
print("        BEST MODEL SUMMARY")
print("=" * 55)
print(f"  Model     : Tuned Random Forest Classifier")
print(f"  Best Params: {grid_search.best_params_}")
print(f"  CV F1     : {grid_search.best_score_:.4f}")
print(f"  Test F1   : {best_f1:.4f}")
print(f"  Test Acc  : {best_acc:.4f}")
print(f"  Precision : {best_prec:.4f}")
print(f"  Recall    : {best_rec:.4f}")
print("─" * 55)
print(f"  Saved files:")
print(f"    • {MODEL_FILE}")
print(f"    • {CLEANED_FILE}")
print(f"    • {SUBMISSION_FILE}")
print("=" * 55)